# 13 - Linearity, `ceiling(gain)` and full well

**Purpose.** Run `protocols/05-linearity.md` and publish `ceiling(gain)` - the level at which the
response departs 1% from a straight line, per CFA plane, per gain - together with the full well in
electrons that `g(gain)` turns it into, and the verdict on whether the bend belongs to the
converter or to the pixel. It is the last bench-only row of MISSION's constants table.

**What it is not for.** Gain, read noise, dark current, PRNU. `g(gain)` is an *input* here, read
from `results/ptc_constants.json` and never re-measured. No pair differencing happens and no
variance is published: the quantity is a **mean level**, and everything this notebook spends its
design on - drift, illumination structure, the offset state, an unbalanced source - is a thing
that ruins a mean.

**One white-ish source does not fill four planes, and that is what shapes this session.** The
panel's spectrum, the Bayer filters and the sensor's QE compose into a different flux per CFA
plane. A ladder scaled to the brightest plane runs the dimmer ones over a fraction of their own
range, so three of the four measurements cannot be made at all. Stretching the ladder does not fix
it: every exposure would scale by the brightest-to-dimmest ratio, which is a property of the bench
that nobody chose. So **the light is balanced instead** - the panel is an LCD, its three subpixels
take three different codes, and gate 4 solves for the triple that makes all four planes collect
the same counts per second. One ladder, four planes, four independent measurements.

**Three halves, and they run at different times.** Section 1 reuses session 02's bench
configuration and re-measures it cold, section 2 captures, section 3 reads the frames back off
disk and publishes. Section 3 needs no camera and no kernel state from above it, so a wrong
analysis costs an afternoon and not a bench night.

## 1. The bench, and the six gates

The bench is session 02's - same panel, same sheet count, same ROI - and this notebook **reuses
that configuration rather than re-scouting it**. `data/session02/bench.json` is a record of a
configuration, not a published constant, and gate 4 below re-measures every number it needs cold.
If the camera or the sheets have been moved since, that file is fiction: re-run section 1 of
`05_ptc.ipynb` first and copy the new `bench.json` here.

Six gates. Two are session 02's and four are new, and they run in this order because each one
needs the one above it:

| gate | what it settles | new? |
|---|---|---|
| 1 | white balance, verified from the pixels | session 02's |
| 2 | the cooler holds at -10 C | session 02's |
| 3 | **the panel's redraw period**, reported by the page itself | new |
| 4 | **the patch colour per gain**, then `t_sat` at it | new |
| 5 | is the light steady in wall clock and in exposure length (L31) | new |
| 6 | the illumination map, which chooses the analysis ROI (L09) | new |

Gates 3 to 5 exist because of one thing session 02 could ignore and this session cannot. A photon
transfer curve plots variance against *measured* signal, so a panel that misbehaves moves a point
along the curve. Linearity plots signal against *commanded exposure*, and a screen that redraws
every 16-odd milliseconds is not a steady lamp on that timescale.

**There is no separate flicker gate, and its absence is deliberate.** Light arriving in redraw
pulses only distorts a ladder if the pulse count is not proportional to the shutter time. Gate 4
refuses to shoot a gain whose faintest rung is under `MIN_RUNG_PERIODS` redraws, so the
arithmetic - one pulse in `N` - bounds that error before a frame is taken; and the one mechanism
the arithmetic does not cover, a slow envelope on the backlight, is exactly what gate 5's second
arm measures. A third gate testing a region the ladder never visits would produce a cut with
nothing to cut, and a cut that fires anyway is a cut that has gone wrong.

In [ ]:
import json
import pathlib
import sys
import time
import urllib.request

import numpy as np
import pandas as pd

sys.path.insert(0, str(pathlib.Path.cwd().parent))
from astropix import asi, fits as F, spatial, stats

ROOT = pathlib.Path.cwd().parent
RESULTS = ROOT / "results"
DATA = ROOT / "data" / "session05"
FRAMES = DATA / "frames"
FRAMES.mkdir(parents=True, exist_ok=True)

FULL_SCALE = 4095                    # ADC counts; the units rule in CLAUDE.md
GAINS = [0, 50, 100, 200]
ROI = (1408, 568, 1024, 1024)        # session 02's ROI: even origin and extent, or Bayer shifts
OFFSET = 15                          # project_offset, fixed by session 01
PATCH_SERVER = "http://127.0.0.1:8765"

# The ladder.  Four geometric rungs to draw the line, sixteen linear ones to
# locate the bend, and the top four are past saturation on purpose.
LINE_RUNGS = [25.0, 31.0, 38.0, 47.0]                          # % of t_sat
BEND_RUNGS = [55.0 + 4.0 * k for k in range(16)]               # 55 .. 115
RUNGS = LINE_RUNGS + BEND_RUNGS
LINE_MAX_PCT = 50.0                  # no rung above this may touch the reference line
MONITOR_PCT = 25.0                   # the drift reference: one between every two ladder frames
BEND_PCT = 1.0                       # the departure that defines the ceiling (L28)

# The panel is a screen, and a screen redraws.  A rung of N redraws carries
# roughly one pulse in N of error, so the ladder is scaled per gain to keep even
# its faintest rung hundreds of redraws long.  22 s of t_sat puts the faintest
# rung near 320 redraws -- 0.3% against a 1% bend.
TARGET_TSAT_S = 22.0
T_SAT_TOL = 0.15                     # fractional; a t_sat this close to target is close enough
MIN_RUNG_PERIODS = 150.0             # a gain whose faintest rung is shorter is not shot at all

# Gate 4's balance.  The four plane fluxes must agree this well: the top rung is
# 115% of t_sat and the dimmest plane must still clear 100% of its own, so a 13%
# imbalance is where the ladder stops saturating it.  5% is that bar with margin.
BALANCE_TOL = 0.05
BALANCE_MAX_ITERS = 8
BALANCE_DAMPING = 0.7                # a full step overshoots: raising one panel channel also
                                     # lifts the planes it crosstalks into
GAMMA_GUESS = 2.2                    # the *first* step only; after that the exponent is measured
CODE_MIN, CODE_MAX = 1, 255
CHANNEL_PLANES = {"r": ["R"], "g": ["G1", "G2"], "b": ["B"]}
PATCH_TIMEOUT_S = 10.0

_bias = json.loads((RESULTS / "bias_constants.json").read_text())
PEDESTAL_FIT = _bias["pedestal_fit"]["value"]
MIN_EXPOSURE = _bias["bias_exposure"]["value"]
HCG = _bias["hcg_threshold_gain"]["value"]

_ptc = json.loads((RESULTS / "ptc_constants.json").read_text())
G_MEASURED = {int(k): v for k, v in _ptc["system_gain"]["value"].items()}
G_ERR = {int(k): v for k, v in _ptc["system_gain"]["uncertainty"].items()}
assert set(GAINS) <= set(G_MEASURED), (
    "every gain here must be one session 02 measured: the gain law's residual is "
    f"{_ptc['gain_law']['value']['residual_pct']}% against its own 1% rule, so g is not "
    "interpolable and a gain with no measured g cannot be turned into electrons")


def amplification(gain):
    """Gain is in 0.1 dB units, so 200 units is exactly a factor of ten."""
    return 10.0 ** (gain / 200.0)


def pedestal_fitted(gain):
    """Session 01's published pedestal at offset 15.  A *prediction* used to
    place a rung; the pedestal every signal is measured against comes from this
    session's own bias block, in section 3."""
    branch = PEDESTAL_FIT["hcg" if gain >= HCG else "lcg"]
    return branch["A"] + branch["B"] * amplification(gain)


def plane_means(mosaic):
    return {k: float(stats.to_adc(v).mean()) for k, v in spatial.split(mosaic).items()}


def panel(path):
    with urllib.request.urlopen(f"{PATCH_SERVER}{path}", timeout=5) as r:
        return json.load(r)


def set_patch(rgb, settle_s=0.4):
    """Drive the panel, and block until the page says it painted.

    The gap between asking and showing is where a rung gets poisoned: the page
    polls every 300 ms and paints on its next redraw, so a capture fired straight
    after `/set` can land on the *previous* colour with nothing in the frame to
    say so.  `/set` returns a seq and the page reports `/applied` with that seq
    once it has painted; this waits for it.  The handshake says the page painted;
    the sleep after it, and the discards the protocol requires, cover the panel
    settling behind that.
    """
    if rgb == "free":
        return panel("/set?rgb=free")
    r, g, b = (int(v) for v in rgb)
    state = panel(f"/set?rgb={r},{g},{b}")
    deadline = time.monotonic() + PATCH_TIMEOUT_S
    while time.monotonic() < deadline:
        applied = panel("/level").get("applied")
        if applied is not None and applied["seq"] >= state["seq"]:
            if applied["rgb"] != [r, g, b]:
                raise RuntimeError(
                    f"asked the panel for {[r, g, b]} and it painted {applied['rgb']}.  Something "
                    "else is driving the page -- close the other notebook or browser tab")
            time.sleep(settle_s)
            return state
        time.sleep(0.1)
    raise RuntimeError(
        f"the page never confirmed patch {[r, g, b]} within {PATCH_TIMEOUT_S:.0f} s.  Reload "
        "grey-patch.html on the iPad -- an older copy has no /applied in it, and this session "
        "must not capture into the gap between asking for a colour and the panel showing it")


def set_grey(level, settle_s=0.4):
    return set_patch([level, level, level], settle_s)


def measure_refresh(seconds=6.0, animated=False):
    """How fast the panel is serving frames, as the page itself reports it.

    `animated` asks the page to animate a 4x4 px dot in its corner.  That is
    not decoration: a display with adaptive refresh serves a *still* page fewer
    frames than the panel drives, so a slow rate with the dot off and a fast
    one with it on means the page was throttled and the panel was not slow.
    Both numbers are recorded, because only the pair is interpretable.
    """
    panel(f"/set?probe={1 if animated else 0}")
    time.sleep(seconds)                       # the page reports every 2 s
    report = panel("/level").get("refresh")
    panel("/set?probe=0")
    if report is None or report["animated"] != animated:
        raise RuntimeError(
            "the page is not reporting its frame rate.  Reload grey-patch.html on the iPad -- "
            "an older copy of the page has no /refresh in it, and this session needs a measured "
            "redraw period rather than an assumed 60 Hz")
    return report


bench = json.loads((ROOT / "data" / "session02" / "bench.json").read_text())
print(f"bench reused from session 02: {bench['sheets']} sheets, "
      f"brightness {bench['brightness_pct']}%, measured {bench['measured_on']}")
print(f"{len(RUNGS)} rungs x {len(GAINS)} gains; g measured at all of them")
print("g(gain), e- per ADC count:  " + "  ".join(f"g{g}={G_MEASURED[g]:.4g}" for g in GAINS))
print()
print("the patch colour is solved per gain in gate 4, not fixed for the session: the subpixels "
      "block light, so the colour moves the four planes against each other without touching the "
      "backlight -- and the panel's own colour shifts with its level, because the backlight leaks "
      "through closed subpixels and the leak is not the colour of the panel driven hard")

# The panel is driven over HTTP, and every gate below assumes it answers.  Fail
# here, naming the command that starts it, rather than three cells down inside a
# gate that has already opened the camera.
try:
    state = panel("/level")
except OSError as exc:
    raise RuntimeError(
        f"{PATCH_SERVER} is not answering ({exc}).  Start protocols/patch-server.py with the "
        "*base* interpreter, not the venv one -- the venv python is a separate binary with no "
        "inbound firewall rule, and the iPad is on the Public network profile:"
        + chr(10) + "  C:/Users/denis/AppData/Local/Python/pythoncore-3.14-64/python.exe "
        "protocols/patch-server.py"
    ) from None
assert "applied" in state, (
    "this patch-server predates the /applied handshake.  Restart it from the current "
    "protocols/patch-server.py, and reload grey-patch.html on the iPad")
print("patch server:", state)

### Gate 1 - white balance, verified from the pixels (L01)

The camera ships `WB_R=55`, `WB_B=75` and applies them to RAW16 before the data reaches us; the
control reading back as 50 proves only that the control took. The evidence is the modal step
between adjacent distinct values: **16 on all four planes**.

It matters more here than anywhere. White balance multiplies red by 1.10 and blue by 1.50 in
integer arithmetic, so a red plane would appear to saturate at 4095/1.10 and a blue one at
4095/1.50 - three different ceilings on one sensor, which is *exactly* the per-plane result this
session exists to measure. A gate 1 failure here does not add noise; it invents the finding.

In [ ]:
# A dead run is restarted, not resumed -- and the check is the whole session
# directory, not just the frames.  A leftover gate4.csv or panel.json from an
# abandoned attempt would be read back by section 3 and quietly paired with
# tonight's pixels, which is a worse failure than a missing file: the gate
# tables say what exposure each rung *was*, so mixing them mislabels every rung.
leftovers = [f for f in DATA.rglob("*") if f.is_file()]
assert not leftovers, (
    f"{len(leftovers)} file(s) already in {DATA}, first {leftovers[0].name} -- delete the whole "
    "directory and start again.  Section 3 reads the gate tables back and pairs them with the "
    "frames by name, so a survivor from an earlier attempt does not fail loudly, it mislabels")

rig = asi.open_camera()
print("gain range     ", rig.range("Gain"))
print("exposure range ", rig.range("Exposure"), "us")
print("white balance shipped:", rig.get("WB_R"), rig.get("WB_B"))

asi.neutralise_white_balance(rig)
asi.set_roi(rig, *ROI)
asi.configure(rig, gain=100, offset=OFFSET)
BIAS_EXPOSURE = rig.min_exposure_s()          # measured, never assumed

set_grey(0)
for _ in range(2):
    asi.capture(rig, BIAS_EXPOSURE)

gate1 = {}
for _ in range(5):
    dark, _ = asi.capture(rig, BIAS_EXPOSURE, imagetyp="DARK")
    for name, plane in spatial.split(dark).items():
        gate1.setdefault(name, []).append(stats.value_step(plane))

steps = {k: sorted(set(v)) for k, v in gate1.items()}
print("modal value step per plane, five frames:", steps)
assert all(v == [16] for v in steps.values()), \
    f"gate 1 FAILED: {steps} -- stop the session, do not correct it later"
print("gate 1 passed")

### Gate 2 - the cooler holds

-10 C, held in band for a continuous 30 seconds before the first frame, judged by the temperature
trend and not by duty cycle. `asi.cool_to` owns the rule and restarts its own clock on any
excursion; every reading is written to disk as it is taken, because the sensor only reports while
we are the ones cooling it.

**The whole session runs in this one kernel.** Closing the camera drops the cooler, and a dead
kernel restarts the cool-down from ambient.

In [ ]:
SETPOINT_C = asi.SETPOINT_C

cool_log = open(DATA / "cooldown.csv", "w", newline="")
cool_log.write("elapsed_s,temp_C,duty_pct" + chr(10))


def show(elapsed, temp, duty):
    cool_log.write(f"{elapsed},{temp},{duty}" + chr(10))
    cool_log.flush()                     # the reading exists nowhere else
    if int(elapsed) % 30 == 0:
        print(f"  {elapsed:6.0f} s  {temp!s:>6} C  {duty:>3}%", flush=True)


try:
    trace = asi.cool_to(rig, SETPOINT_C, log=show)
finally:
    cool_log.close()

temps = [t for _, t, _ in trace if t is not None]
print(f"settled in {trace[-1][0]:.0f} s;  {temps[0]} C -> {temps[-1]} C, minimum {min(temps)} C")
print(f"duty at setpoint {trace[-1][2]}%  <- the headroom this room leaves")

### Gate 3 - the panel's redraw period, measured rather than assumed

"60 Hz" is folklore about a device nobody measured. The page times its own `requestAnimationFrame`
callbacks and reports the median interval, and this reads it back.

**One reading is not enough, and the second one is the point.** A display with adaptive refresh
serves a *still* page fewer frames than the panel actually drives - so a slow rate could mean a
slow panel or a throttled page, and those have opposite consequences. The probe settles it: the
page animates a 4x4 px dot in its corner, which forces a repaint every frame. If the rate rises
with the dot on, the page was being throttled. The dot is black-on-black, four pixels, in a corner
outside any sane ROI, and it is off during every captured frame.

What comes out is `PERIOD_S`, and gate 4 scales the whole ladder against it.

In [ ]:
still = measure_refresh(animated=False)
probed = measure_refresh(animated=True)

print(f"{'':>10} {'period':>10} {'rate':>9} {'p05-p95':>16} {'frames':>7}")
for name, r in (("still", still), ("probed", probed)):
    print(f"{name:>10} {r['period_ms']:9.3f}ms {1000 / r['period_ms']:8.1f}Hz "
          f"{r['p05_ms']:7.3f}-{r['p95_ms']:<7.3f}ms {r['intervals']:7d}")

throttled = probed["period_ms"] < 0.8 * still["period_ms"]
PERIOD_S = min(still["period_ms"], probed["period_ms"]) / 1000.0
jitter = (probed["p95_ms"] - probed["p05_ms"]) / probed["period_ms"]

print()
print("the page was being throttled while still" if throttled else
      "the still page and the probed page are served alike")
print(f"redraw period taken as {PERIOD_S * 1e3:.3f} ms ({1 / PERIOD_S:.1f} Hz) -- the faster of "
      "the two, because the panel cannot drive slower than the frames it delivers")
if jitter > 0.25:
    print(f"WARNING: {100 * jitter:.0f}% spread between the 5th and 95th percentile interval.  "
          "An adaptive-refresh panel that changes rate mid-session makes every rung's phase a "
          "different lottery; gate 5 is the one that decides whether it matters")

### Gate 4 - the patch colour per gain, then `t_sat` at it

**Two knobs, not one.** The *ratios* between the three panel codes decide which plane is
brightest; their *overall scale* decides how long a rung is. They interact, because the panel's
colour shifts with its level - a backlight is always on, light leaks through nominally-closed
subpixels, and the leak is not the colour of the panel driven hard. So the triple is solved **per
gain**, and each gain runs at its own scale and therefore sees its own colour.

The loop, at each gain: capture one frame, read the four plane fluxes, step the three codes,
repeat. Each iteration does two things in order.

1. **Balance, always downward.** Every channel is aimed at the *dimmest* of the three, so the
   step never asks the panel for light it does not have. Balancing down costs flux; step 2 buys it
   back.
2. **Scale.** All three codes move by a common factor to push `t_sat` toward `TARGET_TSAT_S`,
   clipped at code 255. If the panel is already flat out, `t_sat` stays where it is and the
   session is longer - that is an outcome, not a failure.

**It needs no model of the panel, and that is the point.** The subpixel gamma, the Bayer crosstalk
that lets panel-blue reach the red plane, and the backlight leak are all inside the numbers each
iteration reads back. The one number a step needs - how much flux a code is worth *here* - is
measured from the channel's own last two probes rather than assumed; only the very first step uses
a guessed exponent, and it is damped.

**Pass is two numbers:** the four plane fluxes within `BALANCE_TOL` of each other, and the
faintest rung at or above `MIN_RUNG_PERIODS` redraws. A gain that cannot reach both is **dropped
and recorded**, not shot with a ladder nobody can defend.

The probe that measures flux lands the brightest plane between 20% and 70% of headroom - below
full scale on purpose, because a probe placed where the bend lives would be scaled by the very
non-linearity this session is here to measure.

In [ ]:
DISCARD = 2                     # frames dropped after a gain change (protocol)
DISCARD_EXPOSURE = 1            # after an exposure change
DISCARD_PATCH = 2               # after a patch colour change, on top of the handshake


def measure_fluxes(gain, guess_s, tries=5):
    """Counts per second in each of the four planes, at whatever the panel shows.

    The probe walks itself to 20-70% of headroom on the brightest plane.  Below
    full scale on purpose: a probe sitting where the bend lives would be scaled
    by the non-linearity this session exists to measure.
    """
    head, e = FULL_SCALE - pedestal_fitted(gain), guess_s
    sig, exptime = None, None
    for _ in range(tries):
        for _ in range(DISCARD_EXPOSURE):
            asi.capture(rig, e, imagetyp="FLAT")
        mosaic, h = asi.capture(rig, e, imagetyp="FLAT")
        exptime = h["EXPTIME"]
        sig = {p: v - pedestal_fitted(gain) for p, v in plane_means(mosaic).items()}
        top = max(sig.values())
        if 0.2 * head <= top <= 0.7 * head:
            return {p: v / exptime for p, v in sig.items()}, exptime, True
        e = float(np.clip(e * 0.45 * head / max(top, 1.0), MIN_EXPOSURE, 60.0))
    return {p: v / exptime for p, v in sig.items()}, exptime, False


def local_exponent(hist):
    """d log(flux) / d log(code), from this channel's own last two probes.

    The panel's transfer curve is never assumed: two points of its own history
    give the slope where we are standing, which is all a step needs.  Clamped,
    because two nearly-equal codes give a noisy ratio and a wild exponent makes
    a wild step.
    """
    if len(hist) < 2:
        return GAMMA_GUESS
    (c0, f0), (c1, f1) = hist[-2], hist[-1]
    if min(c0, c1, f0, f1) <= 0 or abs(np.log(c1 / c0)) < 1e-3:
        return GAMMA_GUESS
    return float(np.clip(np.log(f1 / f0) / np.log(c1 / c0), 1.0, 4.0))


def balance_gain(gain, start_rgb, guess_s):
    """Solve the patch colour at one gain.  Returns (codes, fluxes, rows)."""
    codes, hist, rows, e = list(start_rgb), {c: [] for c in "rgb"}, [], guess_s
    head = FULL_SCALE - pedestal_fitted(gain)
    fluxes = None

    for it in range(BALANCE_MAX_ITERS):
        set_patch(codes)
        for _ in range(DISCARD_PATCH):
            asi.capture(rig, min(e, 2.0), imagetyp="FLAT")
        fluxes, e, probe_ok = measure_fluxes(gain, e)

        chan = {c: float(np.mean([fluxes[p] for p in ps])) for c, ps in CHANNEL_PLANES.items()}
        for i, c in enumerate("rgb"):
            hist[c].append((codes[i], chan[c]))

        mean_flux = float(np.mean(list(fluxes.values())))
        spread = (max(fluxes.values()) - min(fluxes.values())) / mean_flux
        t_sat_now = head / mean_flux
        faint = RUNGS[0] / 100 * t_sat_now / PERIOD_S
        maxed = max(codes) >= CODE_MAX

        rows.append({"gain": gain, "iter": it, "r": codes[0], "g": codes[1], "b": codes[2],
                     "probe_s": e, **{f"flux_{p}": fluxes[p] for p in spatial.PLANES},
                     "spread_pct": 100 * spread, "t_sat_s": t_sat_now,
                     "faint_rung_periods": faint, "probe_converged": probe_ok})
        print(f"  {it}  rgb {codes[0]:3d},{codes[1]:3d},{codes[2]:3d}  "
              + "  ".join(f"{p}={fluxes[p]:7.1f}" for p in spatial.PLANES)
              + f"  spread {100 * spread:5.2f}%  t_sat {t_sat_now:6.2f}s  "
                f"faint {faint:6.0f} redraws", flush=True)

        # t_sat above target with the panel maxed is as bright as this bench goes,
        # and the protocol accepts it: a longer session, not a worse measurement.
        t_ok = (abs(t_sat_now - TARGET_TSAT_S) / TARGET_TSAT_S <= T_SAT_TOL
                or (t_sat_now > TARGET_TSAT_S and maxed))
        if spread <= BALANCE_TOL and t_ok:
            break

        # 1. Balance downward, to the dimmest channel: always reachable, because
        #    it only ever asks a channel for less light than it is already giving.
        # 2. Scale all three by a factor common *in light*, to move t_sat toward
        #    the target -- and cap that factor so no channel is asked for more
        #    than code 255 can give.
        #
        # The cap is the whole of step 2 and it is not a detail.  Clip the three
        # codes independently instead and the channels with headroom keep rising
        # while the ones at the ceiling cannot, so the panel comes apart at
        # exactly the point where it is working hardest: the balance step 1 just
        # bought is spent by step 2.  Capping the common factor keeps the ratios
        # and gives up only on brightness, which is the thing this session can
        # afford to lose -- a slower t_sat costs session time and nothing else.
        target = min(chan.values())
        exps = [local_exponent(hist[c]) for c in "rgb"]
        scale = t_sat_now / TARGET_TSAT_S

        def codes_for(s, _codes=codes, _chan=chan, _exps=exps, _target=target):
            """Codes that would deliver `_target * s` of light in every channel."""
            return [_codes[i] * float(np.clip(_target * s / max(_chan[c], 1e-9), 0.05, 20.0))
                    ** (BALANCE_DAMPING / _exps[i]) for i, c in enumerate("rgb")]

        for _ in range(4):              # gammas differ, so the binding channel can change
            raw = codes_for(scale)
            over = max(v / CODE_MAX for v in raw)
            if over <= 1.0:
                break
            worst = max(range(3), key=lambda i: raw[i])
            scale *= over ** (-exps[worst] / BALANCE_DAMPING)
        codes = [int(round(float(np.clip(v, CODE_MIN, CODE_MAX)))) for v in codes_for(scale)]

    return codes, fluxes, rows


colour_of, flux, t_sat, balance_pct, gate4 = {}, {}, {}, {}, []
dropped_gains = {}
start = [128, 128, 128]
for g in GAINS:
    print(f"gain {g}", flush=True)
    asi.configure(rig, gain=g, offset=OFFSET)
    for _ in range(DISCARD):
        asi.capture(rig, 0.5, imagetyp="FLAT")
    # Where to put the *first* probe, and nothing else.  Session 02's predicted
    # t_sat places it; the flux that reaches the ladder is measured, never
    # extrapolated (light-source.md item 3).  After the first iteration the probe
    # carries its own exposure forward and this stops mattering.
    guess = (0.25 * list(t_sat.values())[-1] if t_sat else
             0.25 * float(bench.get("t_sat_predicted_s", {}).get(str(g), 2.0)))
    codes, fluxes, rows = balance_gain(g, start, float(np.clip(guess, MIN_EXPOSURE, 60.0)))
    gate4.extend(rows)

    mean_flux = float(np.mean(list(fluxes.values())))
    colour_of[g], flux[g] = codes, mean_flux
    t_sat[g] = (FULL_SCALE - pedestal_fitted(g)) / mean_flux
    balance_pct[g] = 100 * (max(fluxes.values()) - min(fluxes.values())) / mean_flux
    start = codes                                   # the next gain starts where this one landed

    faint = RUNGS[0] / 100 * t_sat[g] / PERIOD_S
    why = []
    if balance_pct[g] > 100 * BALANCE_TOL:
        why.append(f"balance stuck at {balance_pct[g]:.1f}% against {100 * BALANCE_TOL:.0f}%")
    if faint < MIN_RUNG_PERIODS:
        why.append(f"faintest rung {faint:.0f} redraws against {MIN_RUNG_PERIODS:.0f}")
    if why:
        dropped_gains[g] = "; ".join(why)

print()
print(f"{'gain':>5} {'patch':>13} {'flux':>9} {'balance':>9} {'t_sat':>9} {'faint rung':>12}")
for g in GAINS:
    faint = RUNGS[0] / 100 * t_sat[g] / PERIOD_S
    print(f"{g:5d} {','.join(map(str, colour_of[g])):>13} {flux[g]:9.1f} "
          f"{balance_pct[g]:8.2f}% {t_sat[g]:8.2f}s {faint:8.0f} rdw"
          + ("   <- DROPPED: " + dropped_gains[g] if g in dropped_gains else ""))

if dropped_gains:
    print()
    print("these gains are out of reach with this light source and are not shot.  That is "
          "published as a limitation rather than filled with numbers nobody can defend:")
    for g, why in dropped_gains.items():
        print(f"  gain {g}: {why}")
    GAINS = [g for g in GAINS if g not in dropped_gains]
    assert GAINS, "no gain survived gate 4; there is no session to run"

ladder = {g: [r / 100 * t_sat[g] for r in RUNGS] for g in GAINS}
monitor_s = {g: MONITOR_PCT / 100 * t_sat[g] for g in GAINS}
faintest = min(min(v) for v in ladder.values())
assert faintest >= MIN_EXPOSURE, (f"faintest rung {faintest * 1e6:.0f} us is under the "
                                  f"{MIN_EXPOSURE * 1e6:.0f} us shutter floor -- the bench is too "
                                  "bright, and no ladder fixes that")

worst_faint = min(RUNGS[0] / 100 * t_sat[g] / PERIOD_S for g in GAINS)
print()
print(f"every surviving gain clears {MIN_RUNG_PERIODS:.0f} redraws at its faintest rung; worst "
      f"is {worst_faint:.0f}, which is {100 / worst_faint:.2f}% of quantisation against a "
      f"{BEND_PCT}% bend")
print(f"worst plane balance: {max(balance_pct[g] for g in GAINS):.2f}% -- the top rung is "
      f"{RUNGS[-1]:.0f}% of t_sat, so the dimmest plane still clears its own saturation")

N_LADDER, N_BIAS = 3, 10
n_slots = len(RUNGS) * N_LADDER
planned = len(GAINS) * (2 * n_slots + 1 + N_BIAS)
shutter = sum((N_LADDER + DISCARD_EXPOSURE) * sum(ladder[g])
              + (2 * n_slots + 1) * monitor_s[g] for g in GAINS)
print(f"{planned} frames planned, {planned * ROI[2] * ROI[3] * 2 / 1e9:.2f} GB on C:")
print(f"shutter-open {shutter / 60:.0f} min, plus ~{planned * 0.31 / 60:.0f} min of readout: "
      f"{(shutter + planned * 0.31) / 60:.0f} min in all")

pd.DataFrame(gate4).to_csv(DATA / "gate4.csv", index=False)
(DATA / "panel.json").write_text(json.dumps(
    {"period_s": PERIOD_S, "refresh_still": still, "refresh_probed": probed,
     "throttled": bool(throttled),
     # the surviving gains only: section 3 takes its GAINS from this file, and a
     # dropped gain listed here would send the analysis looking for frames that
     # were deliberately never shot
     "patch_of_gain": {str(g): colour_of[g] for g in GAINS},
     "t_sat_of_gain": {str(g): t_sat[g] for g in GAINS},
     "balance_pct_of_gain": {str(g): balance_pct[g] for g in GAINS},
     "dropped_gains": {str(k): v for k, v in dropped_gains.items()},
     "balance_tol": BALANCE_TOL, "min_rung_periods": MIN_RUNG_PERIODS,
     "target_t_sat_s": TARGET_TSAT_S}, indent=2))
print(f"gate 4 written to {DATA / 'gate4.csv'}, panel state to {DATA / 'panel.json'}")

### Gate 5 - is the light steady, in wall clock and in exposure length? (L31)

L31 is the one inherited claim that could invalidate this session outright: two linearity runs
where frame pairs at the same rung should have differed only by noise, and at gain 100 they
differed by **1.79%** against 0.011% at gain 200. An exposure ladder at a fixed light level is
exactly that measurement, so this runs before the ladder and not after it.

Two arms, both at gain 100 - L31's unstable one - on that gain's balanced colour, five minutes
each:

1. **Drift.** One fixed exposure at the monitor rung's 25% of `t_sat`, repeated. Level against
   wall clock. Session 01's dark arm already ran this with the light taken out and was flat to
   -0.00133 +/- 0.254 counts/min, so anything here is upstream of the sensor.
2. **Exposure length against elapsed time.** A short (10%) and a long (45%) exposure alternating
   throughout. Their counts per second should agree; L31's two gains differed in exposure length
   *and* in elapsed time, so their comparison could not separate the two and this one can.

   **Both arms sit inside the reference-line region, and that placement is the whole of their
   validity.** Put the long arm near `t_sat` instead and its level lands where the response may
   already be bending - and a long/short flux ratio can then no longer tell *the light
   short-changes long exposures* from *the sensor is non-linear near full scale*. The second is
   what this session is here to measure, so an arm placed up there answers its own question with
   its own answer. 10% against 45% keeps a 4.5x lever on exposure length, which is all the arm
   needs: a redraw-envelope error goes as one pulse in `N`.

**Arm 2 is also the whole of the redraw worry.** Light arriving in pulses only distorts a ladder
if the pulse count is not proportional to the shutter time, and over rungs of hundreds of redraws
the only mechanism that does that is a slow envelope on the backlight. Comparing counts per second
at 10% and 90% of `t_sat` tests exactly that, and interleaving them is what separates exposure
length from elapsed time.

**A failing gate 5 does not stop the session.** The monitor rungs below divide out drift on any
timescale longer than one frame pair. What gate 5 decides is whether that correction is a safety
net or load-bearing - and that has to be known before the numbers are read, not after.

In [ ]:
STAB_GAIN = 100 if 100 in GAINS else GAINS[len(GAINS) // 2]
STAB_S = 300.0                  # five minutes per arm
# Both arms live *inside the reference-line region*, at or below LINE_MAX_PCT.
# Not at 90%: up there the sensor's own roll-off is in the level, and a long/short
# flux ratio cannot then tell "the light short-changes long exposures" from "the
# response bends at 90% of full scale" -- which is the thing this session exists
# to measure, so the arm would be answering its question with its own answer.
# 10% against 45% keeps a 4.5x lever on exposure length, which is what the arm
# actually needs: a redraw-envelope error goes as one pulse in N.
STAB_SHORT_PCT, STAB_LONG_PCT = 10.0, 45.0

asi.configure(rig, gain=STAB_GAIN, offset=OFFSET)
set_patch(colour_of[STAB_GAIN])
for _ in range(DISCARD_PATCH):
    asi.capture(rig, monitor_s[STAB_GAIN], imagetyp="FLAT")


def stability_arm(name, exposures):
    """Shoot `exposures` round-robin for STAB_S seconds, writing every frame.

    Nothing is discarded and nothing is retaken: the arm is a measurement *of*
    frame-to-frame variation, and a filter on it would remove the signal.
    """
    t0, i = time.monotonic(), 0
    for _ in range(DISCARD):
        asi.capture(rig, exposures[0], imagetyp="FLAT")
    while time.monotonic() - t0 < STAB_S:
        e = exposures[i % len(exposures)]
        mosaic, header = asi.capture(rig, e, imagetyp="FLAT")
        F.write(FRAMES / f"stab_{name}_{i:04d}.fits", mosaic, header)
        i += 1
    print(f"  arm {name}: {i} frames in {(time.monotonic() - t0):.0f} s", flush=True)
    return i


n_drift = stability_arm("drift", [monitor_s[STAB_GAIN]])
n_alt = stability_arm("alt", [STAB_SHORT_PCT / 100 * t_sat[STAB_GAIN],
                              STAB_LONG_PCT / 100 * t_sat[STAB_GAIN]])
print(f"gate 5 captured at gain {STAB_GAIN}: {n_drift} drift frames, {n_alt} alternating.  "
      "Read in section 3.")

### Gate 6 - the illumination map, which chooses the ROI (L09)

L09 says the panel varies **3.8% peak-to-peak across 1024x1024**, so the bright corner saturates
about 4% of exposure before the dim one - smearing a 1% bend over more range than the bend itself.
Central 512 gives 1.25%, central 256 gives 0.53%.

That is a claim about a different bench and it is measured here rather than believed. The rule is
fixed in the protocol: **the largest box whose peak-to-peak is under 0.5%**, and capture stays at
1024 regardless so the choice can be re-made from the same frames in section 3.

**On the balanced colour, and judged on the worst plane.** An LCD's subpixels do not share an
angular response, so a coloured patch can be less uniform across the field than grey, and
differently uniform per channel. The map has to be measured on the light that will actually be
used, and the box has to satisfy every plane rather than the average of them - which is also why
peak-to-peak is measured on a *plane* and never on the mosaic, whose four sensitivities would
otherwise read as illumination structure.

In [ ]:
CROPS = [1024, 512, 256, 128]


def crop_centre(a, n):
    h, w = a.shape
    y, x = ((h - n) // 2) & ~1, ((w - n) // 2) & ~1      # even, or the Bayer phase shifts (L05)
    return a[y:y + n, x:x + n]


def flatness(mosaic, n, side=16):
    """Peak-to-peak of the tile means inside the central `n` box, per plane.

    The tile is a fixed 32x32 mosaic pixels -- 16 a side per plane -- at every
    box, so the shot noise inside one tile does not change as the box does.  Fix
    the tile *count* instead and a small box gets small tiles, the noise in each
    tile mean rises, and the flattest box reads as the worst one.

    One frame, so this is illumination *plus* noise and cannot separate them.
    Section 3 does that from the monitor stacks, and makes the real choice.
    """
    out = {}
    for name, plane in spatial.split(crop_centre(stats.to_adc(mosaic), n)).items():
        k = plane.shape[0] // side
        m = plane[:k * side, :k * side].reshape(k, side, k, side).mean(axis=(1, 3))
        out[name] = float((m.max() - m.min()) / m.mean())
    return out


for _ in range(DISCARD_EXPOSURE):
    asi.capture(rig, 0.5 * t_sat[STAB_GAIN], imagetyp="FLAT")
mosaic, header = asi.capture(rig, 0.5 * t_sat[STAB_GAIN], imagetyp="FLAT")
F.write(FRAMES / "map_000.fits", mosaic, header)

print(f"illumination on the balanced colour {colour_of[STAB_GAIN]} at gain {STAB_GAIN}")
print(f"{'box':>6}  " + "  ".join(f"{p:>8}" for p in spatial.PLANES) + f"  {'worst':>8}   L09 said")
L09_SAID = {1024: 3.8, 512: 1.25, 256: 0.53, 128: None}
worst = {}
for n in CROPS:
    pp = flatness(mosaic, n)
    worst[n] = max(pp.values())
    said = "-" if L09_SAID[n] is None else f"{L09_SAID[n]:.2f}%"
    print(f"{n:>6}  " + "  ".join(f"{100 * pp[p]:7.3f}%" for p in spatial.PLANES)
          + f"  {100 * worst[n]:7.3f}%   {said}")

FLAT_ENOUGH = 0.005
passing = [n for n in CROPS if worst[n] < FLAT_ENOUGH]
first_look = max(passing) if passing else min(CROPS, key=lambda n: worst[n])
print()
print(f"first look says central {first_look}: the largest box under {FLAT_ENOUGH:.1%}, or the "
      "flattest if none passes.  Largest, because among boxes flat enough to trust, the biggest "
      "has the quietest mean")
print(f"a {100 * worst[first_look]:.3f}% spread saturates its bright corner "
      f"{100 * worst[first_look]:.3f}% of exposure early, against a {BEND_PCT}% bend -- and one "
      "frame cannot say how much of that spread is shot noise.  Section 3 decides")
(DATA / "gate6.json").write_text(json.dumps(
    {"peak_to_peak": {str(n): flatness(mosaic, n) for n in CROPS},
     "analysis_box": first_look,
     "patch": colour_of[STAB_GAIN], "gain": STAB_GAIN,
     "rule": f"largest box under {FLAT_ENOUGH}, decided in section 3"}, indent=2))

## 2. The session

**A dead run is restarted, not resumed** - session 01's rule. Delete the whole of
`data/session05/` and go again: the cooler has to settle from scratch anyway, and resumption logic
is code that runs once, in the dark, under time pressure, having never been tested on the case it
exists for.

**The whole directory, not just the frames.** `fits.write` refuses to overwrite, so a surviving
*frame* stops the run loudly. A surviving *gate table* does not: section 3 reads `gate4.csv` and
`panel.json` back and pairs them with the frames by name, so an abandoned attempt's tables would
be silently applied to tonight's pixels and mislabel every rung's exposure **and its patch
colour**. The gate 1 cell therefore refuses to start on any file in the session directory.

**Temperature is enforced by retaking**, as in session 02: a frame whose own header says it was
shot outside the band is not written, and 10 retakes of one slot or a hold past 300 s stops the
session. A retake here can cost a 30-second exposure, so the budget is the same and it binds
harder.

### The patch colour, and the monitor rungs

**Each gain shoots at its own patch colour**, set at the top of its block and put back after the
bias frames, exactly as gate 4 solved it. The monitors move with it, so every correction below is
in that block's own units and no number crosses a colour boundary.

Every colour change goes through the page's handshake and is then followed by two discards. The
handshake says the page painted; the discards cover the panel settling behind it. Neither
substitutes for the other.

Between every ladder frame, one monitor frame at a fixed 25% of `t_sat`. Each rung frame is
divided by the mean of the two monitors bracketing **it**, so drift on any timescale longer than
one frame pair leaves by construction - whatever gate 5 says about its cause. A correction slower
than the thing it corrects is decoration, which is why this brackets frames and not rungs.

In [ ]:
FRAME_GAP_S = 0.2               # readout, USB and the file write are what heat the sensor
HOLD_TIMEOUT_S = 300.0          # a hold that never ends is a cooler fault, not a wait
MAX_RETAKES = 10                # per frame slot; more than this is not a transient
BIAS_PATCH = [0, 0, 0]          # the pedestal block shoots with the panel driven black


def hold_for_temperature(where, exposure_s):
    """Shoot discards until the sensor has been in band for `asi.RECOVER_S`."""
    t0, in_band_since, worst_t = time.monotonic(), None, None
    while True:
        _, header = asi.capture(rig, exposure_s, imagetyp="FLAT")     # discarded on purpose
        time.sleep(FRAME_GAP_S)
        temp, now = header["CCD-TEMP"], time.monotonic()
        worst_t = temp if worst_t is None or (temp is not None and temp > worst_t) else worst_t

        if temp is not None and abs(temp - SETPOINT_C) <= asi.BAND_C:
            in_band_since = now if in_band_since is None else in_band_since
            if now - in_band_since >= asi.RECOVER_S:
                held = now - t0
                print(f"      held {held:.0f} s at {where}, peak {worst_t} C, back at {temp} C",
                      flush=True)
                return held
        else:
            in_band_since = None

        if now - t0 > HOLD_TIMEOUT_S:
            raise TimeoutError(
                f"{HOLD_TIMEOUT_S:.0f} s of holding at {where} and still {temp} C: the cooler is "
                "not keeping up.  Stop the session and check the ambient and the fan -- do not "
                "widen the band")


def capture_frames(n, exposure_s, imagetyp, name, where):
    """`n` in-band frames at one setting, written as `name`_000.fits onwards."""
    retaken, held_s = 0, 0.0
    for i in range(n):
        for _ in range(MAX_RETAKES + 1):
            mosaic, header = asi.capture(rig, exposure_s, imagetyp=imagetyp)
            temp = header["CCD-TEMP"]
            if temp is not None and abs(temp - SETPOINT_C) <= asi.BAND_C:
                break
            retaken += 1
            print(f"    ! {where} frame {i} at {temp} C, retaking", flush=True)
            if temp is not None and temp > SETPOINT_C + asi.BAND_C:
                held_s += hold_for_temperature(where, exposure_s)
            else:
                time.sleep(FRAME_GAP_S)
        else:
            raise RuntimeError(
                f"{MAX_RETAKES} retakes at {where} and still {temp} C.  This is no longer a "
                "transient -- stop the session rather than filling the ladder with frames nobody "
                "can defend")

        F.write(FRAMES / f"{name}_{i:03d}.fits", mosaic, header)
        time.sleep(FRAME_GAP_S)
    return retaken, held_s


def run_gain(gain):
    """One gain: monitor, frame, monitor, frame ... then its own pedestal block.

    **A monitor between every ladder frame, not every rung.** A rung is three
    frames and twenty-odd seconds; the panel moves faster than that, so a bracket
    around a whole rung is slower than the drift it corrects and corrects nothing.
    Here ladder frame `j` sits between monitors `j` and `j+1`, seconds apart, and
    section 3 divides frame by frame.

    It costs one extra frame per ladder frame, at a quarter of `t_sat` each.
    """
    t0, retaken, held = time.monotonic(), 0, 0.0
    asi.configure(rig, gain=gain, offset=OFFSET)
    set_patch(colour_of[gain])         # this gain's own colour, solved in gate 4
    for _ in range(DISCARD + DISCARD_PATCH):
        asi.capture(rig, monitor_s[gain], imagetyp="FLAT")

    slots = [(ri, e) for ri, e in enumerate(ladder[gain]) for _ in range(N_LADDER)]
    last = {}
    for j, (ri, exposure_s) in enumerate(slots):
        for _ in range(DISCARD_EXPOSURE):
            asi.capture(rig, monitor_s[gain], imagetyp="FLAT")
        r, h = capture_frames(1, monitor_s[gain], "FLAT", f"mon_g{gain:03d}_m{j:03d}",
                              f"gain {gain} monitor {j}")
        retaken, held = retaken + r, held + h

        for _ in range(DISCARD_EXPOSURE):
            asi.capture(rig, exposure_s, imagetyp="FLAT")
        i = last.get(ri, -1) + 1
        last[ri] = i
        r, h = capture_frames(1, exposure_s, "FLAT", f"flat_g{gain:03d}_r{ri:02d}_f{i:02d}",
                              f"gain {gain} rung {ri} frame {i} ({exposure_s:.2f} s)")
        retaken, held = retaken + r, held + h
        if i + 1 == N_LADDER:
            print(f"    rung {ri:>2}/{len(RUNGS)}  {RUNGS[ri]:5.1f}%  {exposure_s:8.3f} s  "
                  f"{(time.monotonic() - t0) / 60:5.1f} min", flush=True)

    for _ in range(DISCARD_EXPOSURE):
        asi.capture(rig, monitor_s[gain], imagetyp="FLAT")
    r, h = capture_frames(1, monitor_s[gain], "FLAT", f"mon_g{gain:03d}_m{len(slots):03d}",
                          f"gain {gain} monitor {len(slots)}")     # closes the last bracket
    retaken, held = retaken + r, held + h

    set_patch(BIAS_PATCH)
    for _ in range(DISCARD_PATCH + DISCARD_EXPOSURE):
        asi.capture(rig, BIAS_EXPOSURE, imagetyp="BIAS")
    r, h = capture_frames(N_BIAS, BIAS_EXPOSURE, "BIAS", f"bias_g{gain:03d}",
                          f"gain {gain} pedestal")
    retaken, held = retaken + r, held + h
    set_patch(colour_of[gain])

    print(f"  gain {gain}: {retaken} retaken, {held / 60:.1f} min held, "
          f"{(time.monotonic() - t0) / 60:.1f} min", flush=True)
    return retaken, held

### Blocks 2, 3 and 4 - the ladder, its monitors, and the pedestal beside it

| block | gains | frames |
|---|---|---|
| 2 - ladder | the surviving gains | 20 rungs x 3 |
| 3 - monitor | the surviving gains | 61, interleaved rather than blocked |
| 4 - pedestal | the surviving gains | 10, shot adjacent to that gain's ladder |

**Why the pedestal block matters more here than in session 02.** Session 02 published its `g`
against a bias block whose frames sat in two offset states, and session 11 had to repair it. The
repair is free if it is done first: section 3 runs `stats.offset_state` over this block and takes
the **near-state level**, not the raw mean.

**Why block 4 drives the panel black rather than covering the lens.** A cover is a bench
disturbance, and the attenuation is only valid while nobody moves the camera. The bias exposure is
the shutter floor - microseconds - so whatever still leaks through a black LCD cannot reach these
frames in any amount that matters, and section 3's pedestal table shows it against session 01's
fitted law rather than assuming it.

In [ ]:
t0, retaken, held = time.monotonic(), 0, 0.0
for k, g in enumerate(GAINS, 1):
    print(f"gain {g}  ({k}/{len(GAINS)})  patch {colour_of[g]}", flush=True)
    r, h = run_gain(g)
    retaken, held = retaken + r, held + h
    elapsed = time.monotonic() - t0
    print(f"  == {elapsed / 60:5.1f} min elapsed, "
          f"{elapsed / k * (len(GAINS) - k) / 60:5.1f} min to go", flush=True)

print(f"session: {len(list(FRAMES.glob('*.fits')))} frames, {retaken} retaken, "
      f"{held / 60:.1f} min held, {(time.monotonic() - t0) / 60:.1f} min total")

### Closing down

The cooler is switched off deliberately and the camera closed, and the panel handed back. Let the
sensor warm before unplugging: condensation on a cold sensor is a hardware problem, not a data one.

**Do not move the camera or the sheets until the frame count below is right.** A short session is
recoverable while the bench is still standing and not afterwards.

In [ ]:
rig.set("CoolerOn", 0, verify=False)
rig.close()
set_patch("free")

on_disk = sorted(FRAMES.glob("*.fits"))
print("cooler off, camera closed.  Let it reach ambient before unplugging.")
print(f"{len(on_disk)} frames on disk in {FRAMES}, {planned} planned plus gates 4, 5 and 6")
for prefix in ("flat", "mon", "bias", "stab", "map"):
    print(f"  {prefix:>5}: {sum(1 for f in on_disk if f.name.startswith(prefix))}")

## 3. The analysis

**Everything from here reads disk and nothing else.** The frames, `data/session05/gate4.csv`,
`data/session05/gate6.json`, `data/session05/panel.json`, and session 02's
`results/ptc_constants.json` are the whole input, so the analysis can be re-run and corrected
without ever costing a bench night.

It runs `protocols/05-linearity.md`'s eight analysis rules, fixed before the data existed:

| rule | what it does | published as |
|---|---|---|
| 1 | everything per CFA plane, never on the frame mean | every row of `linearity_rungs.csv` |
| 2 | pedestal from this session's bias block, near state only | `pedestal` column |
| 3 | every rung frame divided by the two monitors bracketing it | `monitor_factor`, `signal` |
| 4 | the reference line fitted through the low rungs, through the origin | `line_slope` |
| 5 | `ceiling` = where the departure reaches 1% | `linearity_constants.json` |
| 6 | a rung with more than 1% of pixels pinned cannot define the bend | `pinned_frac`, `usable` |
| 7 | rules 3-5 re-run at every crop size | `roi` column, and the ROI test |
| 8 | bend levels across gains: the converter or the pixel | the verdict |

In [ ]:
# Section 3 is self-contained on purpose: a fresh kernel can run from here down,
# because the frames and the four files below are the whole input.  Nothing here
# reaches back into a variable the capture half left in memory.
import json
import pathlib
import sys

import numpy as np
import pandas as pd

sys.path.insert(0, str(pathlib.Path.cwd().parent))
from astropix import fits as F, spatial, stats

ROOT = pathlib.Path.cwd().parent
RESULTS = ROOT / "results"
DATA = ROOT / "data" / "session05"
FRAMES = DATA / "frames"

FULL_SCALE = 4095
LINE_RUNGS = [25.0, 31.0, 38.0, 47.0]
BEND_RUNGS = [55.0 + 4.0 * k for k in range(16)]
RUNGS = LINE_RUNGS + BEND_RUNGS
LINE_MAX_PCT, BEND_PCT, MONITOR_PCT = 50.0, 1.0, 25.0
N_LADDER = 3
ROI = (1408, 568, 1024, 1024)
OFFSET = 15

_bias = json.loads((RESULTS / "bias_constants.json").read_text())
_ptc = json.loads((RESULTS / "ptc_constants.json").read_text())
G_MEASURED = {int(k): v for k, v in _ptc["system_gain"]["value"].items()}
G_ERR = {int(k): v for k, v in _ptc["system_gain"]["uncertainty"].items()}
PEDESTAL_FIT, HCG = _bias["pedestal_fit"]["value"], _bias["hcg_threshold_gain"]["value"]


def pedestal_fitted(gain):
    branch = PEDESTAL_FIT["hcg" if gain >= HCG else "lcg"]
    return branch["A"] + branch["B"] * 10.0 ** (gain / 200.0)


RUNGS_CSV = RESULTS / "linearity_rungs.csv"
STABILITY_CSV = RESULTS / "light_stability.csv"
CONSTANTS = RESULTS / "linearity_constants.json"

PLANES = spatial.PLANES
gate6 = json.loads((DATA / "gate6.json").read_text())      # the capture-time first look
gate4 = pd.read_csv(DATA / "gate4.csv")
panel_state = json.loads((DATA / "panel.json").read_text())
PERIOD_S = panel_state["period_s"]
GAINS = sorted(int(g) for g in panel_state["patch_of_gain"])
COLOUR_OF = {int(g): v for g, v in panel_state["patch_of_gain"].items()}
T_SAT = {int(g): v for g, v in panel_state["t_sat_of_gain"].items()}
BALANCE_PCT = {int(g): v for g, v in panel_state["balance_pct_of_gain"].items()}
DROPPED_GAINS = {int(g): v for g, v in panel_state["dropped_gains"].items()}
STAB_GAIN = int(gate6["gain"])

# Rule 6 said "any pinned pixel disqualifies a rung", and on a 1024x1024 ROI that
# is too strict to be usable: a handful of hot pixels pin long before the mean
# nears the top code, and the rung that carries the bend is exactly the one they
# disqualify.  The threshold is a fraction whose bias on the plane mean is below
# the thing being measured -- 1% of pixels clipped drags the mean by well under
# 0.1% of signal, and it drags it *down*, so the ceiling reads low rather than high.
PINNED_MAX = 0.01

# What a ceiling has to clear before it is allowed to be one.  A 1% departure is
# only a measurement if the line it departs from is straighter than that, and if
# the rung it happens on is long enough that light arriving in discrete redraw
# pulses cannot fake it -- one pulse in N, so 0.3% wants 333 redraws.
LINE_RESID_MAX = 0.5      # %, worst rung about the fitted line
QUANT_MAX = 0.003         # one redraw period over the crossing rung's exposure

# There is deliberately no flicker floor here.  Gate 4 refused to shoot any gain
# whose faintest rung was under panel.json's min_rung_periods, so every rung on
# disk is hundreds of redraws long and a cut applied now would have nothing left
# to cut.  The rung length is still published per rung, because the reader is
# owed the number rather than the assurance.
MIN_RUNG_PERIODS = panel_state["min_rung_periods"]

print(f"panel redraw {PERIOD_S * 1e3:.3f} ms, "
      f"{'probed faster than still' if panel_state['throttled'] else 'still and probed agree'}")
print(f"{'gain':>5} {'patch':>13} {'balance':>9} {'t_sat':>9} {'faintest rung':>15}")
for g in GAINS:
    faint = RUNGS[0] / 100 * T_SAT[g] / PERIOD_S
    print(f"{g:5d} {','.join(map(str, COLOUR_OF[g])):>13} {BALANCE_PCT[g]:8.2f}% "
          f"{T_SAT[g]:8.2f}s {faint:11.0f} rdw")
if DROPPED_GAINS:
    print()
    print("gate 4 dropped, and they are published as out of reach rather than estimated:")
    for g, why in DROPPED_GAINS.items():
        print(f"  gain {g}: {why}")


def crop_centre(a, n):
    h, w = a.shape
    y, x = ((h - n) // 2) & ~1, ((w - n) // 2) & ~1
    return a[y:y + n, x:x + n]


def frame_row(path, boxes):
    """One frame at several crop sizes: plane mean and pinned fraction each.

    Pinned is `== FULL_SCALE` in ADC counts.  It is counted per plane and per
    box because rule 6 needs it per plane: under a balanced source the four
    planes should pin together, and a plateau that survives says the balance
    did not hold at the top of the ladder (L12).
    """
    mosaic, header = F.read(path)
    adc = stats.to_adc(mosaic).astype(np.float64)
    out = []
    for n in boxes:
        for name, plane in spatial.split(crop_centre(adc, n)).items():
            out.append({"roi": n, "plane": name, "level": float(plane.mean()),
                        "pinned_frac": float((plane >= FULL_SCALE).mean()),
                        "exptime": float(header["EXPTIME"]),
                        "ccd_temp": header["CCD-TEMP"]})
    return out


BOXES = sorted(int(b) for b in gate6["peak_to_peak"])
print()
print(f"rule 7 re-runs everything at {BOXES}")
print(f"{len(list(FRAMES.glob('*.fits')))} frames to read")

### The analysis box, chosen from the monitor stack (rule 7, L09)

Gate 7 took one frame and printed a first look. This decides, and it does two things that one
frame cannot.

**It measures the noise floor instead of walking into it.** A single flat's tile-to-tile spread is
part illumination and part shot noise, and the noise part *grows* as the box shrinks, because each
tile holds fewer pixels. Judge boxes on that number and the smallest box looks the worst, which is
backwards. So: split each gain-0 monitor stack into odd and even frames, average each, and take
half their difference. That difference contains only noise, and `sqrt(spread^2 - noise^2)` is the
illumination alone. The tile is a fixed **32x32 mosaic pixels** at every box, so nothing changes
underfoot as the box changes.

**And it takes the largest box that passes, not the smallest.** A smaller box is flatter but
noisier: fewer pixels in the plane mean. The flatness test is the constraint - under 0.5%
peak-to-peak - and among boxes that pass it, the biggest is the one with the quietest mean. The
protocol said "smallest" and that was wrong; the rule here is the one that survives being written
down next to its reason.

In [ ]:
TILE_MOSAIC_PX = 32                    # so 16 plane pixels a side, at every box
FLAT_ENOUGH = 0.005


def tile_means(plane, side=TILE_MOSAIC_PX // 2):
    n = plane.shape[0] // side
    return plane[:n * side, :n * side].reshape(n, side, n, side).mean(axis=(1, 3))


def flat_map(paths, box):
    """Illumination spread inside a box, with the shot noise measured and removed."""
    acc = {0: None, 1: None}
    count = {0: 0, 1: 0}
    for i, p in enumerate(paths):
        arr = stats.to_adc(F.read(p)[0]).astype(np.float64)
        acc[i % 2] = arr if acc[i % 2] is None else acc[i % 2] + arr
        count[i % 2] += 1
    a, b = acc[0] / count[0], acc[1] / count[1]
    signal, noise = (a + b) / 2, (a - b) / 2

    out = {}
    for name in spatial.PLANES:
        s = tile_means(spatial.split(crop_centre(signal, box))[name])
        d = tile_means(spatial.split(crop_centre(noise, box))[name])
        p2p = float((s.max() - s.min()) / s.mean())
        floor = float((d.max() - d.min()) / s.mean())
        out[name] = {"p2p": p2p, "noise": floor,
                     "illum": float(np.sqrt(max(p2p ** 2 - floor ** 2, 0.0)))}
    return out


MAP_FRAMES = sorted(FRAMES.glob(f"mon_g{GAINS[0]:03d}_m*.fits"))
flat = {b: flat_map(MAP_FRAMES, b) for b in BOXES}

print(f"{len(MAP_FRAMES)} monitor frames at gain {GAINS[0]}, tiles of "
      f"{TILE_MOSAIC_PX}x{TILE_MOSAIC_PX} mosaic px")
print(f"{'box':>5}  " + "  ".join(f"{p:>8}" for p in spatial.PLANES) +
      f"  {'worst':>8} {'noise':>8}   gate 6 saw")
for b in BOXES:
    worst = max(v["illum"] for v in flat[b].values())
    floor = max(v["noise"] for v in flat[b].values())
    saw = max(gate6["peak_to_peak"][str(b)].values())
    print(f"{b:>5}  " + "  ".join(f"{100 * flat[b][p]['illum']:7.3f}%" for p in spatial.PLANES) +
          f"  {100 * worst:7.3f}% {100 * floor:7.3f}%   {100 * saw:7.3f}%")

passing = [b for b in BOXES if max(v["illum"] for v in flat[b].values()) < FLAT_ENOUGH]
ANALYSIS_BOX = (max(passing) if passing else
                min(BOXES, key=lambda b: max(v["illum"] for v in flat[b].values())))
print()
if passing:
    print(f"analysis box: central {ANALYSIS_BOX} -- the largest under {FLAT_ENOUGH:.1%}, "
          "because among boxes flat enough to trust the biggest has the quietest mean")
else:
    print(f"NO box is under {FLAT_ENOUGH:.1%}: taking the flattest, central {ANALYSIS_BOX}, and "
          "the ceiling below carries that smear -- an illumination spread saturates the bright "
          "corner that fraction of exposure early, against a 1% bend")

### Rule 2 - the pedestal, and the offset state

The pedestal is this session's own block 3, at this gain, **not** session 01's fitted law and not
a bias block from another night: L14's cautionary tale is a dark sitting one count below a bias
shot four hours earlier, which produced a negative dark current.

And it is the **near-state level**, not the block mean. The camera's offset sits in one of two
discrete states separated by a fixed packet of charge at the sense node - about 1.5 e- in low
conversion gain, 0.5 in high - and a bias block that hopped between them has a mean pulled by the
occupancy. `stats.offset_state` is the published classifier for it; session 11 had to apply it to
session 02 after the fact, and doing it first costs one function call.

The occupancy is printed because it is a third night's worth of evidence on how often the far
state is visited, and that is a number two sessions are already arguing about.

In [ ]:
pedestal, occupancy, separation = {}, {}, {}
for g in GAINS:
    files = sorted(FRAMES.glob(f"bias_g{g:03d}_*.fits"))
    levels = {p: [] for p in PLANES}
    for f in files:
        for r in frame_row(f, [ANALYSIS_BOX]):
            levels[r["plane"]].append(r["level"])

    near, far, sep = {}, [], []
    for p in PLANES:
        x = np.asarray(levels[p], float)
        state = stats.offset_state(x)
        # The near state is state 0 -- the populated one -- and the pedestal is
        # its mean.  A group that did not separate comes back all-near, which is
        # the honest answer: no state I can see, at a resolution `separation`
        # None already reports (protocol 04, rule 1).
        near[p] = float(x[~state["far"]].mean())
        far.append(float(state["far"].mean()))
        sep.append(state["separation"])
    pedestal[g] = near
    occupancy[g] = float(np.mean(far))
    separation[g] = [s for s in sep if s is not None]

ped = pd.DataFrame(pedestal).T
ped["far_occupancy"] = pd.Series(occupancy)
ped["state_step"] = pd.Series({g: (np.mean(v) if v else np.nan)
                               for g, v in separation.items()})
ped["fitted_law"] = [pedestal_fitted(g) for g in ped.index]
ped.index.name = "gain"
print(ped.round(4))
print()
print("the fitted_law column is session 01's prediction, not an input: the signal below is "
      "measured against this session's own near-state pedestal")

### Rules 1, 3, 6 and 7 - the rung table

One pass over the ladder, at every crop size, and the only pass: everything below reads this table
rather than the pixels.

**Signal is per plane, per gain, against that gain's near-state pedestal, and every *frame* is
divided by the two monitors that bracket it** - not every rung by the two around the rung. That
was the first run's mistake: the bracket was slower than the wobble it was correcting, so the
monitor factors stayed inside 1% while the rungs moved 2%. A frame here sits seconds from each of
its monitors.

The corrected frames are averaged into the rung afterwards, so `repeat_spread` is the spread of
three *already corrected* frames - which makes it a direct read on whether the correction worked.

In [ ]:
rows = []
for g in GAINS:
    slots = [(ri, i) for ri in range(len(RUNGS)) for i in range(N_LADDER)]
    mon = {}
    for mi in range(len(slots) + 1):
        f = FRAMES / f"mon_g{g:03d}_m{mi:03d}_000.fits"
        for r in frame_row(f, BOXES):
            mon[(mi, r["roi"], r["plane"])] = r["level"] - pedestal[g][r["plane"]]

    grand = {(b, p): float(np.mean([mon[(mi, b, p)] for mi in range(len(slots) + 1)]))
             for b in BOXES for p in PLANES}

    for ri, pct in enumerate(RUNGS):
        files = sorted(FRAMES.glob(f"flat_g{g:03d}_r{ri:02d}_f*.fits"))
        loaded = [(j, frame_row(f, BOXES)) for j, f in
                  ((slots.index((ri, i)), f) for i, f in enumerate(files))]
        for b in BOXES:
            for p in PLANES:
                vals = [(j, r) for j, fr in loaded for r in fr
                        if r["roi"] == b and r["plane"] == p]
                factors = [0.5 * (mon[(j, b, p)] + mon[(j + 1, b, p)]) / grand[(b, p)]
                           for j, _ in vals]
                corrected = [(v["level"] - pedestal[g][p]) / f for (_, v), f in zip(vals, factors)]

                level = float(np.mean([v["level"] for _, v in vals]))
                pinned = float(np.mean([v["pinned_frac"] for _, v in vals]))
                factor = float(np.mean(factors))
                raw = level - pedestal[g][p]
                signal = float(np.mean(corrected))
                spread = float(np.std(corrected, ddof=1))
                exptime = float(np.mean([v["exptime"] for _, v in vals]))
                rows.append({
                    "gain": g, "plane": p, "roi": b, "rung": ri, "rung_pct": pct,
                    "exptime": exptime, "periods": exptime / PERIOD_S,
                    "patch": ",".join(map(str, COLOUR_OF[g])),
                    "level": level, "pedestal": pedestal[g][p],
                    "signal_raw": raw, "monitor_factor": factor, "signal": signal,
                    "repeat_spread": spread, "pinned_frac": pinned,
                    "usable": pinned <= PINNED_MAX,
                    "ccd_temp": float(np.mean([v["ccd_temp"] for _, v in vals])),
                })

rungs = pd.DataFrame(rows)
rungs.to_csv(RUNGS_CSV, index=False)
print(f"{len(rungs)} rows written to {RUNGS_CSV}")

mf = rungs[rungs.roi == ANALYSIS_BOX].groupby("gain").monitor_factor
print()
print("monitor factor per gain (1.0 is a panel that did not move):")
print(pd.DataFrame({"min": mf.min(), "max": mf.max(),
                    "span_pct": 100 * (mf.max() - mf.min())}).round(4))

print()
print(f"shortest rung on disk: {rungs.periods.min():.0f} redraws, against gate 4's floor of "
      f"{MIN_RUNG_PERIODS:.0f} -- so quantisation is at worst "
      f"{100 / rungs.periods.min():.2f}% against a {BEND_PCT}% bend, and no rung is cut for it")

### Rules 4 and 5 - the line, and where the response leaves it

The reference line is fitted through the rungs at or below **50% of `t_sat`** only, and **forced
through the origin in exposure**. A free intercept would absorb exactly the pedestal error that
rule 2 exists to remove, and then report a beautiful straight line through a wrong zero.

`ceiling` is the lowest level whose departure from that line reaches **1%**, interpolated between
the two rungs that bracket it. L28 predicts **3984 counts, 97.3% of the top code**, measured twice
to 0.05% - a prediction to reproduce or refute, never an input.

**Rule 6, and the one place it had to bend.** A clipped rung's mean is compressed by the clip,
which is a different thing from the bend, so clipped rungs are out. But "*any* pinned pixel" is
too strict to be usable on a million-pixel ROI: a handful of hot pixels pin long before the mean
goes near the top code, and the rung that carries the bend is exactly the one that rule throws
away. The threshold is **1% of pixels**, whose bias on the plane mean is far under the 1% being
measured, and which biases the ceiling *down*: conservative in the direction that matters.

**All four planes are on this comparison at equal weight, which is what gate 4 bought.** Under a
white-ish source the ladder is scaled to the brightest plane and the dimmer ones top out at a
fraction of full scale, so three of the four fits have no bend to find. Balanced, every plane
reaches its own saturation on the same rungs - and a plane that still does not is evidence about
the balance, reported as such.

**The bend is searched for only among the bend rungs**, above `LINE_MAX_PCT`, and only where two
consecutive rungs are past the threshold. A first-crossing search from the bottom of the ladder
finds noise and calls it saturation.

In [ ]:
def bend(sub):
    """Rules 4 and 5 for one (gain, plane, roi).  Returns a dict, always."""
    sub = sub.sort_values("exptime")
    low = sub.rung_pct <= LINE_MAX_PCT
    line = sub[low & sub.usable]
    if len(line) < 3:
        return {"line_slope": np.nan, "ceiling": np.nan, "found": False,
                "n_line": len(line), "n_cand": 0, "line_resid_pct": np.nan,
                "quant_pct": np.nan, "top_pinned_frac": float(sub.pinned_frac.max()),
                "reason": "too few rungs left to draw a line"}

    k = float((line.signal * line.exptime).sum() / (line.exptime ** 2).sum())
    resid = 100 * (line.signal / (k * line.exptime) - 1.0)
    sub = sub.assign(dep=sub.signal / (k * sub.exptime) - 1.0)

    cand = sub[~low & sub.usable].reset_index(drop=True)
    below = (cand.dep <= -BEND_PCT / 100).tolist()
    ceiling, found, at = np.nan, False, None
    for i in range(len(cand)):
        # sustained, not a single dip -- except at the very top of the ladder,
        # where the next rung up is the hard clip and there is nothing to sustain
        # into.  Down here the rungs carry thousands of counts, so a 1% departure
        # is tens of counts and not a noise excursion.
        if below[i] and (i + 1 == len(cand) or below[i + 1]):
            at = i
            break

    if at is not None:
        hi = cand.iloc[at]
        # Two precision gates, and a ceiling that fails either is not a ceiling.
        # The line must be straight enough for a 1% departure to mean something,
        # and the crossing rung must be long enough that light arriving in
        # discrete redraw pulses does not quantise it: the error there goes as
        # one pulse in N, so a 100-redraw rung carries 1% before anything else.
        quant = PERIOD_S / hi.exptime
        why = ("line too noisy" if np.abs(resid).max() > LINE_RESID_MAX else
               "exposure too short" if quant > QUANT_MAX else None)
        if why is not None:
            return {"line_slope": k, "ceiling": np.nan, "found": False,
                    "n_line": len(line), "n_cand": len(cand),
                    "line_resid_pct": float(np.abs(resid).max()),
                    "quant_pct": 100 * quant,
                    "top_pinned_frac": float(sub.pinned_frac.max()), "reason": why}
        earlier = sub[(sub.exptime < hi.exptime) & sub.usable]
        lo = earlier.iloc[-1] if len(earlier) else None
        if lo is not None and lo.dep > -BEND_PCT / 100:
            w = (-BEND_PCT / 100 - lo.dep) / (hi.dep - lo.dep)
            ceiling = float((lo.signal + lo.pedestal)
                            + w * ((hi.signal + hi.pedestal) - (lo.signal + lo.pedestal)))
        else:
            ceiling = float(hi.signal + hi.pedestal)
        found = True
    elif len(cand):
        # no sustained departure before the clip: the ceiling is the top code,
        # and the last honest rung is the most this ladder can say
        ceiling = float(cand.iloc[-1].signal + cand.iloc[-1].pedestal)

    # A plane that never approached its own saturation cannot be read as linear
    # to the top code: it simply never went there.  Gate 4 exists to stop this
    # happening, so if it fires the finding is about the balance, not the sensor.
    never_saturated = float(sub.pinned_frac.max()) < PINNED_MAX and not found

    return {"line_slope": k, "ceiling": np.nan if never_saturated else ceiling,
            "found": found, "n_line": len(line), "n_cand": len(cand),
            "line_resid_pct": float(np.abs(resid).max()),
            "quant_pct": 100 * PERIOD_S / cand.iloc[-1].exptime if len(cand) else np.nan,
            "top_pinned_frac": float(sub.pinned_frac.max()),
            "reason": ("ok" if found else
                       "never approached its own saturation -- the balance did not hold"
                       if never_saturated else "no sustained departure before the clip")}


fits_rows = []
for (g, p, b), sub in rungs.groupby(["gain", "plane", "roi"]):
    row = {"gain": g, "plane": p, "roi": b}
    row.update(bend(sub))
    row["pedestal"] = float(sub.pedestal.iloc[0])
    row["signal_at_bend"] = row["ceiling"] - row["pedestal"]
    row["g_e_per_count"] = G_MEASURED[g]
    row["full_well_e"] = row["signal_at_bend"] * G_MEASURED[g]
    fits_rows.append(row)

bends = pd.DataFrame(fits_rows)
bends.loc[~bends.found, ["ceiling", "signal_at_bend", "full_well_e"]] = np.nan
main = bends[bends.roi == ANALYSIS_BOX]
reached = main[main.found]

print(f"why each (gain, plane) did or did not yield a ceiling, central {ANALYSIS_BOX} box:")
print(main.pivot_table(index="gain", columns="plane", values="reason",
                       aggfunc="first").to_string())
print()
print("worst rung about the fitted line, per gain (the 1% bend has to be read against this):")
print(main.groupby("gain").line_resid_pct.max().round(3).to_string())
print()
print("top-rung pinned fraction per plane -- balanced, the four should pin together:")
print(main.pivot_table(index="gain", columns="plane", values="top_pinned_frac",
                       aggfunc="first").round(3).to_string())

if len(reached):
    print()
    print(f"{BEND_PCT}% departure, per gain, over the planes that got there:")
    print(reached.groupby("gain").agg(
        planes=("plane", lambda s: "+".join(sorted(s))),
        ceiling=("ceiling", "mean"),
        plane_spread_pct=("ceiling", lambda s: 100 * (s.max() - s.min()) / s.mean()),
        full_well_e=("full_well_e", "mean"),
        line_resid_pct=("line_resid_pct", "max")).round(3))
    print()
    print(f"L28 predicted 3984 counts ({100 * 3984 / FULL_SCALE:.1f}% of the top code); "
          f"measured {reached.ceiling.mean():.1f} over {len(reached)} (gain, plane) fits")
else:
    print()
    print("NO ceiling is measurable from this dataset, and that is the result.")
    print(f"A {BEND_PCT}% departure cannot be read off a ladder whose own rungs scatter by more "
          "than that about their own straight line.  Everything else this session measured still "
          "stands -- the redraw period, the balance solve, the stability trace, the illumination "
          "map -- and `ceiling` is published as not measured, with the reason.")

unbalanced = main[main.reason.str.startswith("never approached")]
if len(unbalanced):
    print()
    print("planes that never reached their own saturation, which gate 4 exists to prevent:")
    print(unbalanced[["gain", "plane", "top_pinned_frac"]].to_string(index=False))
    print("this is a statement about the balance at the top of the ladder, not about the sensor")

### Rule 7 - the ROI test (L09)

L09's rule was *use a small ROI for linearity*, and its reason was that uneven illumination smears
the bend over more range than the bend itself. That is a claim with a measurable consequence: the
bend should move as the box grows, and in one direction - a wider box mixes in corners that
saturate earlier, so the departure arrives sooner and the ceiling comes out **low**.

If the ceiling is flat across boxes, L09's rule is not wrong but it is not load-bearing here
either, and that is worth publishing as plainly as the ceiling is.

In [ ]:
roi_test = pd.DataFrame(index=pd.Index(BOXES, name="roi"))
roi_test["illum_pct"] = [100 * max(v["illum"] for v in flat[b].values()) for b in BOXES]
roi_test["line_resid_pct"] = [bends[bends.roi == b].line_resid_pct.max() for b in BOXES]

if bends.found.any():
    got = bends[bends.found].groupby("roi").ceiling
    roi_test["ceiling"] = got.mean()
    roi_test["n"] = got.size()
    roi_test["vs_analysis_box_pct"] = 100 * (roi_test.ceiling / roi_test.ceiling[ANALYSIS_BOX] - 1)
    print(roi_test.round(4))
    print()
    print(f"the ceiling moves {roi_test.vs_analysis_box_pct.abs().max():.3f}% across boxes from "
          f"{min(BOXES)} to {max(BOXES)} px, against a {BEND_PCT}% bend definition")
    print("L09 predicts the wide box reads low; a flat column says the smearing is not what "
          "limits this measurement")
else:
    roi_test["vs_analysis_box_pct"] = np.nan
    print(roi_test.round(4))
    print()
    print("no box yielded a ceiling, so L09's claim is untested here.  What the columns do show "
          "is the ordering L09 predicted -- illumination spread rises with box size - and that "
          "the line residual barely moves with it, which says the box is not what limits this "
          "measurement.  Something common to every box is, and gate 5 names it.")

### Rule 8 - the converter, or the pixel

The one structural question this session can answer. A bend that belongs to the **ADC** sits at a
fixed *level* in counts, at every gain and on every plane. A bend that is the **pixel well
filling** sits at a fixed *charge*, so its level in counts is `Q/g` and rises steeply with gain -
until it runs into the top code and the ADC binds instead.

Both are published, and the verdict is whichever is flat. Per-plane spread is the yardstick: L12
found all four planes bending at one level to 1.6%, which is what made "the converter bends, not
the pixel" a reading rather than a guess.

In [ ]:
per_gain = reached.groupby("gain").agg(
    ceiling=("ceiling", "mean"),
    ceiling_sd=("ceiling", "std"),
    n_planes=("plane", "size"),
    plane_spread_pct=("ceiling", lambda s: 100 * (s.max() - s.min()) / s.mean()),
    signal_at_bend=("signal_at_bend", "mean"),
    full_well_e=("full_well_e", "mean"))
per_gain["pct_of_top_code"] = 100 * per_gain.ceiling / FULL_SCALE
per_gain["Q_over_g_counts"] = per_gain.full_well_e / pd.Series(G_MEASURED).loc[per_gain.index]

if len(per_gain) >= 2:
    flat_in_counts = float(100 * (per_gain.ceiling.max() - per_gain.ceiling.min())
                           / per_gain.ceiling.mean())
    flat_in_charge = float(100 * (per_gain.full_well_e.max() - per_gain.full_well_e.min())
                           / per_gain.full_well_e.mean())
    plane_spread = float(per_gain.plane_spread_pct.max())
    verdict = "converter" if flat_in_counts < flat_in_charge else "pixel well"
    print(per_gain.round(3))
    print()
    print(f"ceiling varies {flat_in_counts:.2f}% across gains in ADC counts")
    print(f"full well varies {flat_in_charge:.2f}% across gains in electrons")
    print(f"worst per-plane spread at one gain: {plane_spread:.2f}%")
    print()
    print(f"verdict: the bend follows the {verdict} -- whichever is flat is what bends")
else:
    flat_in_counts = flat_in_charge = plane_spread = float("nan")
    verdict = None
    print(f"{len(per_gain)} gain(s) yielded a ceiling; the converter-or-pixel question needs at "
          "least two and is not answered by this dataset.  It is published as null, not guessed.")

### Gate 5 read back - the L31 stability trace

Two arms, and the second is the one the retired project could not run. Arm 1 is drift against wall
clock at a fixed exposure. Arm 2 alternates a short and a long exposure throughout, so **exposure
length and elapsed time are separated** rather than confounded - and its long/short flux ratio is
this session's whole test of whether light arriving in redraw pulses reaches the sensor in
proportion to the shutter time. A ratio of 1.0 means it does.

In [ ]:
def arm_table(pattern):
    out = []
    for f in sorted(FRAMES.glob(pattern)):
        r = [x for x in frame_row(f, [ANALYSIS_BOX]) if x["plane"] == "G1"][0]
        _, header = F.read(f)
        out.append({"file": f.name, "t": header["DATE-OBS"], "exptime": r["exptime"],
                    "level": r["level"] - pedestal[STAB_GAIN]["G1"]})
    d = pd.DataFrame(out)
    d["minutes"] = (pd.to_datetime(d.t) - pd.to_datetime(d.t).iloc[0]).dt.total_seconds() / 60
    d["flux"] = d.level / d.exptime
    return d


drift = arm_table("stab_drift_*.fits")
alt = arm_table("stab_alt_*.fits")

slope, _ = np.polyfit(drift.minutes, drift.level, 1)
slope_err = float(drift.level.std(ddof=1) / np.sqrt(len(drift)) / max(drift.minutes.max(), 1e-9))
drift_pct = 100 * drift.level.std(ddof=1) / drift.level.mean()
print(f"arm 1, drift: {len(drift)} frames over {drift.minutes.max():.1f} min at gain {STAB_GAIN}")
print(f"  slope {slope:+.4f} +/- {slope_err:.4f} counts/min "
      f"({100 * slope / drift.level.mean():+.4f} %/min), "
      f"frame-to-frame scatter {drift_pct:.3f}%")
print("  session 01's dark arm: -0.00133 +/- 0.254 counts/min, with the light taken out")

alt["arm"] = np.where(alt.exptime > alt.exptime.median(), "long", "short")
by_arm = alt.groupby("arm").agg(n=("flux", "size"), exptime=("exptime", "mean"),
                                periods=("exptime", lambda s: s.mean() / PERIOD_S),
                                flux=("flux", "mean"),
                                scatter_pct=("flux", lambda s: 100 * s.std(ddof=1) / s.mean()))
length_ratio = float(by_arm.flux["long"] / by_arm.flux["short"])
# Read off the frames rather than carried down from the capture half: section 3
# is self-contained, and what the arms actually shot is the honest label anyway.
arm_pct = {a: 100 * by_arm.exptime[a] / T_SAT[STAB_GAIN] for a in ("short", "long")}
print()
print("arm 2, exposure length against elapsed time:")
print(by_arm.round(4))
print(f"  long/short flux ratio {length_ratio:.4f}")
print("  1.0 means the measured flux does not depend on how long the shutter was open -- which "
      "is also the whole of the redraw question, since both arms are hundreds of redraws long")
print()
print("L31's contrast to beat: 1.79% at gain 100 against 0.011% at gain 200")

stability = pd.concat([drift.assign(arm="drift"), alt], ignore_index=True)
stability.to_csv(STABILITY_CSV, index=False)
print(f"written to {STABILITY_CSV}")

### Publishing

Two CSVs are already written - `linearity_rungs.csv` (every rung, plane and crop size) and
`light_stability.csv` (gate 6's two arms). This cell writes the scalars with their provenance.

`ceiling` is published **per gain and per plane**, because the star-colour constraint binds on the
brightest plane and a single number would hide that. `full_well_e` carries `g(gain)`'s own
uncertainty into its own: the ceiling is a level this session measured, the electrons are session
02's scale applied to it, and a reader is entitled to know which half of the product moved.

In [ ]:
on_disk = sorted(FRAMES.glob("*.fits"))
measured_on = str(F.read(on_disk[0])[1]["DATE-OBS"])[:10]
n_frames = len(on_disk)


def constant(value, unit, uncertainty, note):
    return {"value": value, "unit": unit, "uncertainty": uncertainty,
            "source_frames": n_frames, "measured_on": measured_on,
            "notebook": "13_linearity.ipynb", "note": note}


def nn(v):
    """None for anything that is not a finite number: a null is a published
    statement that the quantity was not measured, and NaN is not JSON."""
    return None if v is None or not np.isfinite(v) else round(float(v), 4)


ceiling_by_gain = {int(g): nn(v) for g, v in per_gain.ceiling.items()}
well_by_gain = {int(g): nn(v) for g, v in per_gain.full_well_e.items()}
well_err = {int(g): nn(per_gain.full_well_e[g] * G_ERR[g] / G_MEASURED[g])
            for g in per_gain.index}

constants = {
    "ceiling": constant(
        ceiling_by_gain, "ADC counts",
        {int(g): nn(v) for g, v in per_gain.ceiling_sd.items()},
        f"rule 5 of protocols/05-linearity.md: the level at which the response departs "
        f"{BEND_PCT}% from a line fitted through the rungs below {LINE_MAX_PCT}% of t_sat and "
        f"forced through the origin.  Mean over the four CFA planes in the central "
        f"{ANALYSIS_BOX} box; per-plane values are in linearity_rungs.csv and the fits per "
        f"(gain, plane, roi) are reproducible from it.  Uncertainty is the per-plane scatter.  "
        f"L28 predicted 3984 counts, 97.3% of the top code"),
    "ceiling_per_plane": constant(
        {int(g): {p: nn(v) for p, v in s.set_index("plane").ceiling.items()}
         for g, s in main.groupby("gain")},
        "ADC counts", None,
        "the constraint binds per plane: sky flux differs per CFA channel, so the exposure "
        "floor is set by the dimmest plane and the clipping ceiling by the brightest (MISSION).  "
        "Gate 4 balanced the source so all four planes reach their own saturation on the same "
        "ladder; a null here means that plane still did not, which is a statement about the "
        "balance and not about the sensor"),
    "not_measured": constant(
        {f"{int(r.gain)}/{r.plane}": r.reason for r in main.itertuples() if not r.found},
        "reason per (gain, plane)", None,
        "every fit that did not yield a ceiling, and why.  A published null with a reason is the "
        "point of this entry: the alternative is a number that looks like a measurement"),
    "full_well": constant(
        well_by_gain, "e-", well_err,
        "(ceiling - pedestal) x g(gain), with g consumed from ptc_constants.json and never "
        "re-measured here.  The uncertainty is g's alone: the ceiling's own scatter is the "
        "separate ceiling entry, and the two are not independent enough to add"),
    "bend_follows": constant(
        verdict, "one of: converter, pixel well", None,
        f"rule 8.  The ceiling varies {flat_in_counts:.2f}% across gains in ADC counts and the "
        f"full well varies {flat_in_charge:.2f}% in electrons; whichever is flat is what bends.  "
        f"Worst per-plane spread at one gain is {plane_spread:.2f}%, against L12's 1.6%.  "
        f"null means fewer than two gains yielded a ceiling, so the question was not answered"),
    "roi_sensitivity": constant(
        {int(b): nn(v) for b, v in roi_test.vs_analysis_box_pct.items()},
        "% change in ceiling against the analysis box", None,
        f"rule 7 / L09.  The analysis box is the central {ANALYSIS_BOX}: the *largest* whose "
        f"illumination spread is under {FLAT_ENOUGH:.1%}, measured on the gain-{GAINS[0]} monitor "
        "stack with the shot-noise floor subtracted (odd frames against even).  Largest, because "
        "among boxes flat enough to trust the biggest has the quietest mean.  L09 predicted 3.8% "
        "at 1024, 1.25% at 512 and 0.53% at 256, and claimed a wide box reads the ceiling low"),
    "panel_redraw_period": constant(
        PERIOD_S * 1e3, "ms", 1.0,
        "gate 3: the page timed its own requestAnimationFrame callbacks and reported the median "
        "interval, still and with a 4x4 px corner dot animating.  "
        + ("the probed rate was faster, so the page was being throttled and the probed period is "
           "the one used" if panel_state["throttled"] else "still and probed agree") +
        ".  It is the rate frames are *served*, which bounds the panel from below and is not a "
        "reading of the backlight -- what the backlight does is gate 5 arm 2, with the camera"),
    "patch_colour_per_gain": constant(
        {int(g): COLOUR_OF[g] for g in GAINS}, "panel RGB codes 0-255", None,
        "gate 4: the subpixel codes that make the four CFA planes collect equal flux at that "
        "gain, solved by measured iteration from the plane fluxes themselves -- no panel gamma, "
        "crosstalk matrix or backlight-leak model is assumed anywhere.  Solved per gain because "
        "the panel's colour shifts with its level: the backlight leaks through closed subpixels "
        "and the leak is not the colour of the panel driven hard.  This is also the per-plane "
        "sensitivity of this bench under this source, read directly rather than inferred from "
        "pinned-pixel plateaus (L12).  It is a property of *this panel*, not of the camera"),
    "plane_balance": constant(
        {int(g): nn(BALANCE_PCT[g]) for g in GAINS}, "% spread across the four plane fluxes", None,
        f"gate 4's pass criterion, at {100 * panel_state['balance_tol']:.0f}%.  The top rung is "
        f"{RUNGS[-1]:.0f}% of t_sat and the dimmest plane must still clear 100% of its own, so a "
        "13% imbalance is where the ladder stops saturating it; this is how much margin there "
        "actually was"),
    "t_sat_per_gain": constant(
        {int(g): nn(T_SAT[g]) for g in GAINS}, "s", None,
        f"gate 4: the exposure that fills the headroom above the pedestal at the balanced colour, "
        f"target {panel_state['target_t_sat_s']:.0f} s.  Every rung is a percentage of it.  A "
        f"gain further from target than the rest is one where the panel was flat out, which costs "
        f"session time and nothing else"),
    "gains_out_of_reach": constant(
        {int(g): why for g, why in DROPPED_GAINS.items()}, "reason per gain", None,
        f"gate 4 shoots a gain only if its four planes balance to "
        f"{100 * panel_state['balance_tol']:.0f}% and its faintest rung clears "
        f"{MIN_RUNG_PERIODS:.0f} redraws -- one pulse in N, so that is "
        f"{100 / MIN_RUNG_PERIODS:.2f}% of quantisation against a {BEND_PCT}% bend.  A gain that "
        f"cannot is published here rather than shot with a ladder nobody can defend"),
    "light_drift": constant(
        nn(slope), "ADC counts per minute", nn(slope_err),
        f"gate 5 arm 1 (L31): {len(drift)} frames at gain {STAB_GAIN} over "
        f"{drift.minutes.max():.1f} minutes at a fixed exposure, panel warm.  Session 01's dark "
        "arm measured -0.00133 +/- 0.254 counts/min with the light taken out, so anything here "
        "is upstream of the sensor"),
    "flux_vs_exposure_length": constant(
        nn(length_ratio),
        f"ratio of counts/s at {arm_pct['long']:.0f}% t_sat to counts/s at "
        f"{arm_pct['short']:.0f}%",
        None,
        "gate 5 arm 2 (L31): the arm the retired project could not run, because their two gains "
        "differed in exposure length and in elapsed time at once.  1.0 means the measured flux "
        "does not depend on how long the shutter was open -- which is also this session's whole "
        "test of the redraw worry, since light arriving in pulses can only distort a ladder if "
        "the pulse count is not proportional to the shutter time.  Both arms sit inside the "
        "reference-line region on purpose: near t_sat the sensor's own roll-off is in the level, "
        "and the ratio could no longer separate a light that short-changes long exposures from a "
        "response that bends near full scale -- which is the quantity this session publishes.  "
        "Scatter: "
        f"{by_arm.scatter_pct['short']:.3f}% short, {by_arm.scatter_pct['long']:.3f}% long, "
        f"against L31's 1.79% at this gain"),
}

CONSTANTS.write_text(json.dumps(constants, indent=2))
print(f"written to {CONSTANTS}")
print()
for k, v in constants.items():
    print(f"  {k:<24} {json.dumps(v['value'])[:78]}")
print()
print(f"{n_frames} frames, captured {measured_on}")
print(f"also written: {RUNGS_CSV.name}, {STABILITY_CSV.name}")